In [1]:
import qgrid
import requests
import pandas as pd
import json
import datetime as dt

In [2]:
def binance_data(symbol, interval, startTime, endTime, chunk_size=1000):
    url = "https://api.binance.com/api/v3/klines"
    df_list = []
    
    start_time = startTime
    while start_time < endTime:
        end_time = start_time + pd.DateOffset(days=1)  # Adjust as needed
        end_time = min(end_time, endTime)
        
        start_time_ms = int(start_time.timestamp() * 1000)
        end_time_ms = int(end_time.timestamp() * 1000)
        
        req_params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": str(start_time_ms),
            "endTime": str(end_time_ms),
            "limit": str(chunk_size)
        }
        
        response = requests.get(url, params=req_params)
        if response.status_code == 200:
            data = json.loads(response.text)
            if len(data) > 0:
                df = pd.DataFrame(data)
                df = df.iloc[:, 0:6]
                df.columns = ["opentime", "open", "high", "low", "close", "volume"]
                df.index = [dt.datetime.fromtimestamp(x / 1000.0) for x in df.opentime]
                df.drop(columns=['opentime'], inplace=True)  # Drop the opentime column
                df.insert(0, "time", df.index)  # Insert calculated time column
                df_list.append(df)
        
        start_time = end_time
    
    if not df_list:
        return None
    
    final_df = pd.concat(df_list)
    return final_df


In [3]:
df = binance_data("MATICUSDT","5m",dt.datetime(2023,8,26),dt.datetime(2023,8,28))

In [4]:
df

,time,open,high,low,close,volume
2023-08-26 00:00:00,2023-08-26 00:00:00,0.54390000,0.54480000,0.54360000,0.54470000,109136.20000000
2023-08-26 00:05:00,2023-08-26 00:05:00,0.54480000,0.54580000,0.54480000,0.54530000,105167.80000000
2023-08-26 00:10:00,2023-08-26 00:10:00,0.54540000,0.54560000,0.54500000,0.54540000,93936.50000000
2023-08-26 00:15:00,2023-08-26 00:15:00,0.54550000,0.54610000,0.54520000,0.54590000,94622.00000000
2023-08-26 00:20:00,2023-08-26 00:20:00,0.54600000,0.54780000,0.54600000,0.54740000,254094.30000000
...,...,...,...,...,...,...
2023-08-28 05:10:00,2023-08-28 05:10:00,0.55540000,0.55560000,0.55520000,0.55520000,32731.80000000
2023-08-28 05:15:00,2023-08-28 05:15:00,0.55530000,0.55620000,0.55520000,0.55610000,91575.40000000
2023-08-28 05:20:00,2023-08-28 05:20:00,0.55600000,0.55600000,0.55560000,0.55580000,19376.40000000
2023-08-28 05:25:00,2023-08-28 05:25:00,0.55580000,0.55610000,0.55580000,0.55590000,12064.60000000


In [6]:
df.to_csv("matic5m.csv",index=False)